In [ ]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
import healpy as hp
import heracles
import heracles.dices as dices
from heracles.io import read

In [ ]:
from astropy.io import fits
# Open the FITS file
config_path = "scripts/sims_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
mask_type = config['mask_type']
hdul = fits.open(f"/home/jaimerzp/Documents/UCL/GLASS_cov_challenge/{mask_type}/cls_pols/kernels_pols_1.fits")

# Print information about the file
hdul.info()

In [ ]:
plt.imshow(np.log10(np.abs(hdul[0].data[0, :, :].T)),  cmap='seismic')
plt.colorbar()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 5))
plt.subplots_adjust(hspace = 0.0, wspace = 0.0)

im0 = axs[0, 0].imshow(np.log10(np.abs(hdul[0].data[0, :, :].T)), cmap='seismic')
axs[0, 0].get_xaxis().set_ticks([])
axs[0, 0].set_title('POS x POS x POS x POS', y=0.8)

im1 = axs[0, 1].imshow(np.log10(np.abs(hdul[0].data[1, :, :].T)), cmap='seismic')
axs[0, 1].get_yaxis().set_ticks([])
axs[0, 1].get_xaxis().set_ticks([])
axs[0, 1].set_title('POS x E x POS x B', y=0.8)

im2 = axs[1, 0].imshow(np.log10(np.abs(hdul[0].data[2, :, :].T)), cmap='seismic')
axs[1, 0].set_title('E x E x B x B', y=0.8)

im3 = axs[1, 1].imshow(np.log10(np.abs(hdul[0].data[3, :, :].T)), cmap='seismic')
axs[1, 1].get_yaxis().set_ticks([])
axs[1, 1].set_title('B x B x E x E', y=0.8)

fig.subplots_adjust(right=0.8)
cbar_ax = fig.add_axes([0.82, 0.15, 0.02, 0.7])
fig.colorbar(im0, cax=cbar_ax)
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/inv_kernel_compo_{mask_type}.pdf', bbox_inches='tight')

## Comparison

In [ ]:
config_path = "scripts/sims_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
n = config['nsims']
nside = config['nside']
lmax = config['lmax']
mode = config['mode']  # "lognormal" or "gaussian"
mask_type = config['mask_type']  # Default to 'Patch' if not specified
binned = False #True
apply_mask = True
nbins = 3

output_path = f"{mode}_dices/"
output_path = "./masked_"+output_path

nlbins = config.get('nlbins', 20)  # Default to 20 if not specified
l = np.arange(lmax+1)
ls = np.arange(lmax+1)
ledges = np.logspace(np.log10(10), np.log10(lmax), nlbins + 1)
lgrid = (ledges[1:] + ledges[:-1]) / 2

In [ ]:
def get_cls_mean(cls_dict):
    n_keys = list(cls_dict.keys())
    f_keys = list(cls_dict[n_keys[0]].keys())
    cls_mean = {}
    for f_key in f_keys:
        cl = np.mean([cls_dict[i][f_key] for i in n_keys], axis=0)
        cls_mean[f_key] = heracles.Result(cl, axis=cls_dict[n_keys[0]][f_key].axis)
    return cls_mean

In [ ]:
theory_cls = heracles.read(f"lognormal_sims/cls_theory.fits")
ls = np.arange(lmax+1)
fl = -np.sqrt((ls+2)*(ls+1)*ls*(ls-1))
fl /= np.clip(ls*(ls+1), 1, None)

_theory_cls = {}
_theory_cls[("POS", "POS", 1, 1)] = heracles.Result(theory_cls["W1xW1"].array[:lmax+1], ell=ls)

c = np.zeros((2, 2, lmax+1))
c[0, 0, :] = theory_cls["W2xW2"].array[:lmax+1] * fl**2
_theory_cls[("SHE", "SHE", 1, 1)] = heracles.Result(c)

c = np.zeros((2, lmax+1))
c[0, :] = theory_cls["W1xW2"].array[:lmax+1] * fl
_theory_cls[("POS", "SHE", 1, 1)] = heracles.Result(c)

_theory_cqs = heracles.binned(_theory_cls, ledges)

In [ ]:
#path = "./dummy/"
#cls = {}
#for i in range(1, n+1):
#    print(f"Loading sim {i}", end='\r')
#    cls[i] = heracles.read(path+f"/cls/cls_data_{i}.fits")

#cqs = heracles.binned(cls, ledges)
#_theory_cls = get_cls_mean(cls)
#_theory_cqs = get_cls_mean(cqs)

In [ ]:
path = f"./{mask_type}/"
cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    cls[i] = heracles.read(path+f"/cls/cls_data_{i}.fits")

cqs = heracles.binned(cls, ledges)
cls_m = get_cls_mean(cls)
cqs_m = get_cls_mean(cqs)

In [ ]:
# Create a figure with 1 row and 3 columns of subplots
fig, axs = plt.subplots(1, 3, figsize=(15, 4))  # 1 row, 3 columns

# POS POS
axs[0].plot(l[2:], l[2:]*cls_m[("POS", "POS", 1, 1)].array[2:], color='C0', label='PP')
axs[0].plot(l[2:], l[2:]*_theory_cls[("POS", "POS", 1, 1)].array[2:], color='C0', linestyle='--')
axs[0].legend()
axs[0].set_title("POS-POS")
axs[0].set_xscale('log')
axs[0].set_yscale('log')
axs[0].set_xlabel("l")
axs[0].set_ylabel("C_l")

# POS SHE
axs[1].plot(l[2:], l[2:]*cls_m[("POS", "SHE", 1, 1)].array[0][2:], color='C0', label='PE')
axs[1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[0][2:], color='C0', linestyle='--')
axs[1].plot(l[2:], l[2:]*cls_m[("POS", "SHE", 1, 1)].array[1][2:], color='C1', label='PB')
axs[1].plot(l[2:], l[2:]*_theory_cls[("POS", "SHE", 1, 1)].array[1][2:], color='C1', linestyle='--')
axs[1].legend()
axs[1].set_title("POS-SHE")
axs[1].set_xscale('log')
axs[1].set_yscale('symlog')
axs[1].set_xlabel("l")

# SHE SHE
axs[2].plot(l[2:], l[2:]*cls_m[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', label='EE')
axs[2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 0][2:], color='C0', linestyle='--')
axs[2].plot(l[2:], l[2:]*cls_m[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', label='EB')
axs[2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[0, 1][2:], color='C2', linestyle='--')
axs[2].plot(l[2:], l[2:]*cls_m[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', label='BB')
axs[2].plot(l[2:], l[2:]*_theory_cls[("SHE", "SHE", 1, 1)].array[1, 1][2:], color='C1', linestyle='--')
axs[2].legend()
axs[2].set_title("SHE-SHE")
axs[2].set_xscale('log')
axs[2].set_yscale('log')
#axs[2].set_yscale('symlog', linthresh=5e-10)
axs[2].set_xlabel("l")

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
path = f"./{mask_type}/"
inv_cls = {}
nu_cls = {}
pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    inv_cls[i] = heracles.read(path+f"/cls_inv/cls_data_inv_{i}.fits")
    nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}.fits")
    pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}.fits")

In [ ]:
nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}.fits")
inv_cqs = heracles.binned(inv_cls, ledges)
nu_cqs = heracles.binned(nu_cls, ledges)
pols_cqs = heracles.binned(pols_cls, ledges)

In [ ]:
inv_cls_m = get_cls_mean(inv_cls)
nu_cls_m = get_cls_mean(nu_cls)
pols_cls_m = get_cls_mean(pols_cls)

inv_cqs_m = get_cls_mean(inv_cqs)
nu_cqs_m = get_cls_mean(nu_cqs)
pols_cqs_m = get_cls_mean(pols_cqs)
nmt_cqs_m = get_cls_mean(nmt_cqs)

In [ ]:
ensemble_cov = heracles.read(f"{mask_type}/covs/cov_cls.fits")
ensemble_covqq = heracles.read(f"{mask_type}/covs/cov_cqs.fits")

In [ ]:
inv_ensemble_cov = heracles.read(f"{mask_type}/covs/cov_inv_cls.fits")
nu_ensemble_cov = heracles.read(f"{mask_type}/covs/cov_nu_cls.fits")
pols_ensemble_cov = heracles.read(f"{mask_type}/covs/cov_pols_cls.fits")

inv_ensemble_covqq = heracles.read(f"{mask_type}/covs/cov_inv_cqs.fits")
nu_ensemble_covqq = heracles.read(f"{mask_type}/covs/cov_nu_cqs.fits")
nmt_ensemble_covqq = heracles.read(f"{mask_type}/covs/cov_nmt_cqs.fits")
pols_ensemble_covqq = heracles.read(f"{mask_type}/covs/cov_pols_cqs.fits")


In [ ]:
# Flattened ensemble Covariance
flat_ensemble_cov = dices.flatten(ensemble_covqq)
flat_ensemble_corr = flat_ensemble_cov / np.sqrt(
    np.diag(flat_ensemble_cov)[:, None] * np.diag(flat_ensemble_cov)[None, :]
)
# Flattened inverse Covariance
flat_inv_ensemble_cov = dices.flatten(inv_ensemble_covqq)
flat_inv_ensemble_corr = flat_inv_ensemble_cov / np.sqrt(
    np.diag(flat_inv_ensemble_cov)[:, None] * np.diag(flat_inv_ensemble_cov)[None, :]
)
# Flattened nu ensemble Covariance
flat_nu_ensemble_cov = dices.flatten(nu_ensemble_covqq)
flat_nu_ensemble_corr = flat_nu_ensemble_cov / np.sqrt(
    np.diag(flat_nu_ensemble_cov)[:, None] * np.diag(flat_nu_ensemble_cov)[None, :]
)
# Flattened PolSpice plus Covariance
#flat_pp_ensemble_cov = dices.flatten(pp_ensemble_covqq)
#flat_pp_ensemble_corr = flat_pp_ensemble_cov / np.sqrt(
#    np.diag(flat_pp_ensemble_cov)[:, None] * np.diag(flat_pp_ensemble_cov)[None, :]
#)
# Flattened PolSpice minus Covariance
#flat_pm_ensemble_cov = dices.flatten(pm_ensemble_covqq)
#flat_pm_ensemble_corr = flat_pm_ensemble_cov / np.sqrt(
#    np.diag(flat_pm_ensemble_cov)[:, None] * np.diag(flat_pm_ensemble_cov)[None, :]
#)
# Flattened PolSpice polarization Covariance
flat_pols_ensemble_cov = dices.flatten(pols_ensemble_covqq)
flat_pols_ensemble_corr = flat_pols_ensemble_cov / np.sqrt(
    np.diag(flat_pols_ensemble_cov)[:, None] * np.diag(flat_pols_ensemble_cov)[None, :]
)
# Flattened PolSpice decoupled polarization Covariance
#flat_dpols_ensemble_cov = dices.flatten(dpols_ensemble_covqq)
#flat_dpols_ensemble_corr = flat_dpols_ensemble_cov / np.sqrt(
#    np.diag(flat_dpols_ensemble_cov)[:, None] * np.diag(flat_dpols_ensemble_cov)[None, :]
#)
# Flattened NMT Covariance
flat_nmt_ensemble_cov = dices.flatten(nmt_ensemble_covqq)
flat_nmt_ensemble_corr = flat_nmt_ensemble_cov / np.sqrt(
    np.diag(flat_nmt_ensemble_cov)[:, None] * np.diag(flat_nmt_ensemble_cov)[None, :]
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 7))

# Flattened Ensemble Covariance
im1 = axes[0, 0].imshow(flat_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 0].set_title("Ensemble")

# Flattened Inverse covariance
im2 = axes[0, 1].imshow(flat_inv_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 1].set_title("Inverse")

# Flattened NMT Covariance
im3 = axes[0, 2].imshow(flat_nmt_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
axes[0, 2].set_title("NaMaster")

# Flattened Natural unmixing Covariance
im4 = axes[1, 0].imshow(flat_nu_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 0].set_title("Natural Unmixing")

# Flattened PolSpice Plus Covariance
im5 = axes[1, 1].imshow(flat_pols_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
axes[1, 1].set_title("PolSpice PolSpice")

# Flattened PolSpice Minus Covariance
#im6 = axes[1, 2].imshow(flat_dpols_ensemble_corr, cmap="seismic", vmin=-1, vmax=1)
#axes[1, 2].set_title("PolSpice Decoupled PolSpice")
#fig.suptitle(f'Correlation Matrices for {mask_type} Mask', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(5, 8))

# First plot: Eigenvalue relative difference
eig_ens_sorted = np.sort(np.linalg.eigvalsh(flat_ensemble_cov))[::-1]
eig_inv_ens_sorted = np.sort(np.linalg.eigvalsh(flat_inv_ensemble_cov))[::-1]
eig_nmt_ens_sorted = np.sort(np.linalg.eigvalsh(flat_nmt_ensemble_cov))[::-1]
eig_nu_ens_sorted = np.sort(np.linalg.eigvalsh(flat_nu_ensemble_cov))[::-1]
eig_pols_ens_sorted = np.sort(np.linalg.eigvalsh(flat_pols_ensemble_cov))[::-1]
#eig_dpols_ens_sorted = np.sort(np.linalg.eigvalsh(flat_dpols_ensemble_cov))[::-1]
ens_err = np.sqrt(np.diag(flat_ensemble_cov))
inv_ens_err = np.sqrt(np.diag(flat_inv_ensemble_cov))
nmt_ens_err = np.sqrt(np.diag(flat_nmt_ensemble_cov))
nu_ens_err = np.sqrt(np.diag(flat_nu_ensemble_cov))
pols_ens_err = np.sqrt(np.diag(flat_pols_ensemble_cov))
#dpols_ens_err = np.sqrt(np.diag(flat_dpols_ensemble_cov))
axes[0].plot(1 - eig_inv_ens_sorted/eig_ens_sorted, label='Inverse')
axes[0].plot(1 - eig_nmt_ens_sorted/eig_ens_sorted, label='NaMaster')
axes[0].plot(1 - eig_nu_ens_sorted/eig_ens_sorted, label='Natural Unmixing')
axes[0].plot(1 - eig_pols_ens_sorted/eig_ens_sorted, label='PolSpice')
#axes[0].plot(1 - eig_dpols_ens_sorted/eig_ens_sorted, label='PolSpice (decoupled)')
axes[0].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[0].set_yscale('symlog', linthresh=1e1)
axes[0].set_xlabel('Eigenvalue index (sorted)')
axes[0].set_ylabel('Eigenvalue relative difference')
axes[0].set_title(f'{mask_type} Mask')
axes[0].legend()

# Second plot: Diagonal of Covariance Matrices
axes[1].plot((inv_ens_err-ens_err)/ens_err, label='Inverse')
axes[1].plot((nmt_ens_err-ens_err)/ens_err, label='NaMaster')
axes[1].plot((nu_ens_err-ens_err)/ens_err, label='Natural Unmixing')
axes[1].plot((pols_ens_err-ens_err)/ens_err, label='PolSpice')
#axes[1].plot((dpols_ens_err-ens_err)/ens_err, label='PolSpice (decoupled)')
axes[1].axhline(0, color='k', linestyle='--', linewidth=0.5)
axes[1].set_ylabel('Square root variance relative difference')
axes[1].set_xlim(0, 120)
axes[1].set_xticks(np.arange(20, 121, 20))
axes[1].set_xticklabels(["PxP", "PxE", "PxB", "ExE", "ExB", "BxB"])
axes[1].legend()

plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/cov_comp_{mask_type}.pdf', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(12, 8), gridspec_kw={"height_ratios": [3, 1, 3, 1]})
fig.subplots_adjust(left=0.0, bottom=0.0, right=1.0, top=1.0, wspace=0.15, hspace=0.0)

# POS
key = ("POS", "POS", 1, 1)
cov_key = ("POS", "POS", "POS", "POS", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key]
nmt_cov = nmt_ensemble_covqq[cov_key]
nu_cov = nu_ensemble_covqq[cov_key]
pp_cov = pols_ensemble_covqq[cov_key]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key]
nmt_c = nmt_cqs_m[key]
nu_c = nu_cqs_m[key]
pp_c = pols_cqs_m[key]
t = _theory_cls[key]
t_itp = np.interp(lgrid, ls, t)


ax[0, 0].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 0].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[0, 0].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 0].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (nmt_c - t_itp) / nmt_err,
    yerr=np.abs(nmt_err / nmt_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[0, 0].tick_params(axis="both", which="both", direction="in")

ax[0, 0].set_title("POSxPOS", y=0.9)
ax[0, 0].set_xscale("log")
ax[0, 0].set_xlim(5, lmax * 2)
ax[0, 0].set_ylabel("$\ell C_\ell$")
ax[1, 0].set_yscale("symlog", linthresh=5e-1)
ax[1, 0].set_ylim(-100, 100)
ax[1, 0].set_xscale("log")
ax[1, 0].set_xlim(5, lmax * 2)
ax[1, 0].set_xlabel("$\ell$")
ax[1, 0].set_ylabel(r"$\frac{\ell \Delta C_\ell}{\sigma_{C_\ell}}$")

# POS-E
key = ("POS", "SHE", 1, 1)
cov_key = ("POS", "SHE", "POS", "SHE", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key][0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, :, :]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, :]
nmt_c = nmt_cqs_m[key][0, :]
nu_c = nu_cqs_m[key][0, :]
pp_c = pols_cqs_m[key][0, :]
t = _theory_cls[key][0, :]
t_itp = np.interp(lgrid, ls, t)

ax[0, 1].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 1].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[0, 1].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 1].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 1].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 1].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[0, 1].tick_params(axis="both", which="both", direction="in")
ax[1, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)

ax[0, 1].set_title("POSxE", y=0.9)
ax[0, 1].set_xscale("log")
ax[0, 1].set_xlim(5, lmax * 2)
ax[1, 1].set_yscale("symlog", linthresh=5e-1)
ax[1, 1].set_ylim(-100, 100)
ax[1, 1].set_xscale("log")
ax[1, 1].set_xlim(5, lmax * 2)
ax[1, 1].set_xlabel(r"$\ell$")

# SHE-B
key = ("POS", "SHE", 1, 1)
cov_key = ("POS", "SHE", "POS", "SHE", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key][1, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][1, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][1, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][1, 1, :, :]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][1, :]
nmt_c = nmt_cqs_m[key][1, :]
nu_c = nu_cqs_m[key][1, :]
pp_c = pols_cqs_m[key][1, :]
t = _theory_cls[key][1, :]
t_itp = np.interp(lgrid, ls, t)

ax[0, 2].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[0, 2].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[0, 2].plot(ls[10:], t[10:], c="k", lw=1.0, zorder=4.0)
ax[1, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[1, 2].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 2].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[1, 2].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[0, 2].legend()
ax[0, 2].set_title("POSxB", y=0.9)
ax[1, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[0, 2].tick_params(axis="both", which="both", direction="in")
ax[0, 2].set_xscale("log")
ax[0, 2].set_xlim(5, lmax * 2)
ax[1, 2].set_yscale("symlog", linthresh=5e-1)
ax[1, 2].set_ylim(-100, 100)
ax[1, 2].set_xscale("log")
ax[1, 2].set_xlim(5, lmax * 2)
ax[1, 2].set_xlabel(r"$\ell$")

# EE
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 0, 0, 0, :, :]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, 0, :]
nmt_c = nmt_cqs_m[key][0, 0, :]
nu_c = nu_cqs_m[key][0, 0, :]
pp_c = pols_cqs_m[key][0, 0, :]
t = _theory_cls[key][0, 0, :]
t_itp = np.interp(lgrid, ls, t)


ax[2, 0].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 0].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
    )
ax[2, 0].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 0].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (nmt_c - t_itp) / nmt_err,
    yerr=np.abs(nmt_err / nmt_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 0].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C3", 
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)

ax[2, 0].tick_params(axis="both", which="both", direction="in")
ax[3, 0].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[2, 0].set_ylabel("$\ell C_\ell$")
ax[2, 0].set_title("ExE", y=0.9)
ax[2, 0].set_xscale("log")
ax[2, 0].set_xlim(5, lmax * 2)
ax[3, 0].set_ylabel(r"$\frac{\ell \Delta C_\ell}{\sigma_{\ell \Delta C_\ell}}$")
ax[3, 0].set_yscale("symlog", linthresh=5e-1)
ax[3, 0].set_ylim(-100, 100)
ax[3, 0].set_xscale("log")
ax[3, 0].set_xlim(5, lmax * 2)
ax[3, 0].set_xlabel(r"$\ell$")

# EB
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key][0, 1, 0, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][0, 1, 0, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][0, 1, 0, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][0, 1, 0, 1, :, :]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][0, 1, :]
nmt_c = nmt_cqs_m[key][0, 1, :]
nu_c = nu_cqs_m[key][0, 1, :]
pp_c = pols_cqs_m[key][0, 1, :]
t = _theory_cls[key][0, 1, :]
t_itp = np.interp(lgrid, ls, t)

ax[2, 1].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 1].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[2, 1].plot(ls[10:], ls[10:]*t[10:], c="k", lw=1.0, zorder=4.0)
ax[3, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 1].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].errorbar(
    lgrid,
    (nmt_c - t_itp) / nmt_err,
    yerr=np.abs(nmt_err / nmt_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C3",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 1].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[2, 1].tick_params(axis="both", which="both", direction="in")
ax[2, 1].set_title("ExB", y=0.9)
ax[2, 1].set_xscale("log")
ax[2, 1].set_xlim(5, lmax * 2)
ax[3, 1].set_yscale("symlog", linthresh=5e-1)
ax[3, 1].set_ylim(-100, 100)
ax[3, 1].set_xscale("log")
ax[3, 1].set_xlim(5, lmax * 2)
ax[3, 1].set_xlabel(r"$\ell$")

# BB
key = ("SHE", "SHE", 1, 1)
cov_key = ("SHE", "SHE", "SHE", "SHE", 1, 1, 1, 1)

i_cov = inv_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nmt_cov = nmt_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
nu_cov = nu_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]
pp_cov = pols_ensemble_covqq[cov_key][1, 1, 1, 1, :, :]

i_err = np.sqrt(np.diag(i_cov))
nmt_err = np.sqrt(np.diag(nmt_cov))
nu_err = np.sqrt(np.diag(nu_cov))
pp_err = np.sqrt(np.diag(pp_cov))

i_c = inv_cqs_m[key][1, 1, :]
nmt_c = nmt_cqs_m[key][1, 1, :]
nu_c = nu_cqs_m[key][1, 1, :]
pp_c = pols_cqs_m[key][1, 1, :]
t = _theory_cls[key][1, 1, :]
t_itp = np.interp(lgrid, ls, t)

ax[2, 2].errorbar(
            lgrid, lgrid*i_c, yerr=lgrid*i_err, fmt=".", c="C0", lw=1.5, zorder=3.0, alpha=0.5,
            label="Inversion"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*nmt_c, yerr=lgrid*nmt_err, fmt=".", c="C1", lw=1.5, zorder=3.0, alpha=0.5,
    label="NaMaster"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*nu_c, yerr=lgrid*nu_err, fmt=".", c="C2", lw=1.5, zorder=3.0, alpha=0.5,
    label="Natural Unmixing"
        )
ax[2, 2].errorbar(
    lgrid, lgrid*pp_c, yerr=lgrid*pp_err, fmt=".", c="C3", lw=1.5, zorder=3.0, alpha=0.5,
    label="PolSpice"
        )
ax[2, 2].plot(ls[10:], t[10:], c="k", lw=1.0, zorder=4.0, label="Theory")
ax[3, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[3, 2].errorbar(
    lgrid,
    (i_c - t_itp) / i_err,
    yerr=np.abs(i_err / i_err),
    fmt=".",
    c="C0",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].errorbar(
    lgrid,
    (nmt_c - t_itp) / nmt_err,
    yerr=np.abs(nmt_err / nmt_err),
    fmt=".",
    c="C1",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].errorbar(
    lgrid,
    (nu_c - t_itp) / nu_err,
    yerr=np.abs(nu_err / nu_err),
    fmt=".",
    c="C2",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[3, 2].errorbar(
    lgrid,
    (pp_c - t_itp) / pp_err,
    yerr=np.abs(pp_err / pp_err),
    fmt=".",
    c="C3",
    lw=1.5,
    zorder=1.0,
    alpha=0.5,
)
ax[2, 2].set_title("BxB", y=0.9)
ax[3, 2].axhline(0.0, c="k", lw=0.8, zorder=-1)
ax[2, 2].tick_params(axis="both", which="both", direction="in")
ax[2, 2].set_xscale("log")
ax[2, 2].set_xlim(5, lmax * 2)
ax[3, 2].set_yscale("symlog", linthresh=5e-1)
ax[3, 2].set_ylim(-100, 100)
ax[3, 2].set_xscale("log")
ax[3, 2].set_xlim(5, lmax * 2)
ax[3, 2].set_xlabel(r"$\ell$")
fig.suptitle(f"Cls comparison for {mask_type} mask", fontsize=16, y=1.05)
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/cls_compo_{mask_type}.pdf', bbox_inches='tight')

In [ ]:
def get_xi2s(cls_data, cls_theory, covs):
    """Calculate the chi-squared value."""
    xi2s = {}
    for key in list(cls_data.keys()):
        a, b, i, j = key
        covkey = (a, b, a, b, i, j, i, j)
        cov = covs[covkey]
        cl_data = cls_data[key]
        cl_theory = cls_theory[key]
        if a == b == "POS":
            cl_data = cl_data.array
            cl_theory = cl_theory.array
            diff = cl_data - cl_theory
            cov = cov.array
            invcov = np.linalg.pinv(cov)
            xi2 = np.dot(diff, np.dot(invcov, diff))
            xi2s[key] = xi2
        elif a == b == "SHE":
            cl_data_ee = cl_data[0, 0, :]
            cl_data_eb = cl_data[0, 1, :]
            cl_data_bb = cl_data[1, 1, :]
            cl_theory_ee = cl_theory[0, 0, :]
            cl_theory_eb = cl_theory[0, 1, :]
            cl_theory_bb = cl_theory[1, 1, :]
            diff_ee = cl_data_ee - cl_theory_ee
            diff_eb = cl_data_eb - cl_theory_eb
            diff_bb = cl_data_bb - cl_theory_bb
            cov_ee = cov[0, 0, 0, 0, :, :]
            cov_eb = cov[0, 1, 0, 1, :, :]
            cov_bb = cov[1, 1, 1, 1, :, :]
            invcov_ee = np.linalg.pinv(cov_ee)
            invcov_eb = np.linalg.pinv(cov_eb)
            invcov_bb = np.linalg.pinv(cov_bb)
            xi2_ee = np.dot(diff_ee, np.dot(invcov_ee, diff_ee))
            xi2_eb = np.dot(diff_eb, np.dot(invcov_eb, diff_eb))
            xi2_bb = np.dot(diff_bb, np.dot(invcov_bb, diff_bb))
            xi2s[('E', 'E', i, j)] = xi2_ee
            xi2s[('E', 'B', i, j)] = xi2_eb
            xi2s[('B', 'B', i, j)] = xi2_bb
        elif a == "POS" and b == "SHE":
            cl_pe = cl_data[0, :]
            cl_pb = cl_data[1, :]
            cl_theory_pe = cl_theory[0, :]
            cl_theory_pb = cl_theory[1, :]
            diff_pe = (cl_pe - cl_theory_pe)
            diff_pb = (cl_pb - cl_theory_pb)
            cov_pe = cov[0, 0, :, :]
            cov_pb = cov[1, 1, :, :]
            invcov_pe = np.linalg.pinv(cov_pe)
            invcov_pb = np.linalg.pinv(cov_pb)
            xi2_pe = np.dot(diff_pe, np.dot(invcov_pe, diff_pe))
            xi2_pb = np.dot(diff_pb, np.dot(invcov_pb, diff_pb))
            xi2s[('POS', 'E', i, j)] = xi2_pe
            xi2s[('POS', 'B', i, j)] = xi2_pb
        else:
            raise ValueError(f"Unknown key: {key}")
    return xi2s

In [ ]:
xi2s = get_xi2s(cqs_m, _theory_cqs, inv_ensemble_covqq)
xi2s_inv = get_xi2s(inv_cqs_m, _theory_cqs, inv_ensemble_covqq)
xi2s_nmt = get_xi2s(nmt_cqs_m, _theory_cqs, nmt_ensemble_covqq)
xi2s_nu = get_xi2s(nu_cqs_m, _theory_cqs, nu_ensemble_covqq)
xi2s_pols = get_xi2s(pols_cqs_m, _theory_cqs, pols_ensemble_covqq)

In [ ]:
import matplotlib.pyplot as plt

# List of chi-squared dictionaries to plot
chi2_dicts = {
    "Inverse": xi2s_inv,
    "NaMaster": xi2s_nmt,
    "Natural Unmixing": xi2s_nu,
    "PolSpice": xi2s_pols,
}

keys = list(xi2s.keys())
labels = [
    "POSxPOS", "POSxE", "POSxB",
    "ExE", "ExB", "BxB"
]

x = range(len(keys))
width = 0.13

fig, ax = plt.subplots(figsize=(6, 4))

for i, (name, d) in enumerate(chi2_dicts.items()):
    values = [d[k] for k in keys]
    ax.bar(
        [xi + i * width for xi in x],
        values,
        width=width,
        label=name
    )

ax.set_xticks([xi + width*2.5 for xi in x])
ax.set_xticklabels(labels)
ax.set_ylabel(r"$\chi^2$")
ax.set_title(f"{mask_type} Mask")
ax.set_yscale("log")
ax.legend()
plt.tight_layout()
plt.show()
fig.savefig(f'/home/jaimerzp/Desktop/mixing_mat_plots/Xi2s_{mask_type}.pdf', bbox_inches='tight')